In [ ]:
%pip install pandas scikit-learn pytesseract Pillow

In [ ]:
import pandas as pd
import re
import pytesseract
from PIL import Image
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB


data = {
    "Message": [


        "Onam Celebration in College Auditorium tomorrow 10 AM",
        "Annual Sports Day next Monday at 9 AM",
        "Cultural Fest registrations open from 10th Sep",
        "Hackathon event this weekend",
        "Independence Day celebration on 15th Aug at 8 AM",
        "Farewell party for final years on 30th March 6 PM",
        "College Day function scheduled on 5th April 10 AM",
        "Blood Donation Camp in Seminar Hall on 20th July",
        "Alumni Meet on 25th December at 11 AM",
        "Christmas Celebration at College Auditorium on 23rd Dec",
        "Sports Meet closing ceremony on 28th Jan 3 PM",
        "Republic Day parade on 26th Jan at 8:30 AM",

        "Join AI Tech Talk on 10th Sep at 3 PM",
        "Web Development Workshop on Friday 5 PM",
        "Data Science Seminar via Zoom on 12th Sep",
        "Cyber Security Workshop on 15th Sep 2025 at 11:30 AM",
        "Cloud Computing Seminar on 2nd Nov 10 AM",
        "Android App Development Workshop on 18th Feb at 2 PM",
        "AI Ethics Webinar on 25th March at 4 PM",
        "Robotics Seminar on 7th July at 10:30 AM",
        "IoT Workshop scheduled on 10th Aug at 9 AM",
        "Blockchain Seminar on 15th Oct 11 AM",
        "Machine Learning Guest Lecture on 5th Dec 3 PM",
        "Big Data Analytics Workshop on 20th Jan 2 PM",


        "Auditions for College Drama on 15th Sep 2 PM",
        "Coding Competition registration opens 12th Sep",
        "Music Competition on 18th Sep at 6 PM",
        "Dance Auditions on 22nd Nov at 4 PM",
        "Quiz Competition on 1st Dec 10 AM",
        "Poster Making Competition on 12th Feb 9 AM",
        "Hackathon registrations open till 25th March",
        "Photography Contest submission deadline 30th June",
        "Debate Competition on 10th July 11 AM",
        "Poetry Recital on 15th August 3 PM",
        "Drama Club Auditions on 5th Oct 2 PM",
        "Gaming Tournament on 18th Dec at 5 PM",

        "Assignment 2 submission deadline is tomorrow",
        "Library closed today due to maintenance",
        "Exam timetable released for 3rd semester",
        "Submit assignment 3 by next Wednesday",
        "Midterm exam schedule published",
        "Final exam hall tickets available online",
        "Project report submission deadline 10th April",
        "Revaluation applications open till 25th May",
        "Practical exam dates announced for 2nd year",
        "Lab manuals submission last date 1st March",
        "Holiday declared on 2nd October for Gandhi Jayanti",
        "Online class links shared in student portal",
        "Result publication for 4th semester on 15th June",
        "Attendance shortage list released for verification"
    ],
    "Label": [

        "Event","Event","Event","Event","Event","Event",
        "Event","Event","Event","Event","Event","Event",


        "Tech Talk","Tech Talk","Tech Talk","Tech Talk",
        "Tech Talk","Tech Talk","Tech Talk","Tech Talk",
        "Tech Talk","Tech Talk","Tech Talk","Tech Talk",


        "Audition","Audition","Audition","Audition",
        "Audition","Audition","Audition","Audition",
        "Audition","Audition","Audition","Audition",


        "Other Academic","Other Academic","Other Academic","Other Academic",
        "Other Academic","Other Academic","Other Academic","Other Academic",
        "Other Academic","Other Academic","Other Academic","Other Academic",
        "Other Academic","Other Academic"
    ]
}

df = pd.DataFrame(data)


vectorizer = TfidfVectorizer(stop_words="english")
X = vectorizer.fit_transform(df["Message"])
y = df["Label"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
model = MultinomialNB()
model.fit(X_train, y_train)
print(" Model trained!\n")


def extract_text_from_image(image_path):
    text = pytesseract.image_to_string(Image.open(image_path))
    text = re.sub(r'[\n\r]+', ' ', text)
    text = re.sub(r'[^\w\s:/.,-]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def extract_datetime(message):
    date_patterns = [
        r'\d{1,2}(st|nd|rd|th)?\s?(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[\s,]*\d{0,4}',
        r'(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)\s\d{1,2},?\s?\d{2,4}?',
        r'\d{1,2}/\d{1,2}/\d{2,4}',
        r'\d{4}-\d{1,2}-\d{1,2}',
        r'\d{1,2}\s?(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)'
    ]

    time_patterns = [
        r'\d{1,2}(:\d{2})?\s?(AM|PM)',
        r'\d{1,2}(:\d{2})'
    ]

    date = "Not found"
    time = "Not found"

    for pattern in date_patterns:
        match = re.search(pattern, message, re.IGNORECASE)
        if match:
            date = match.group(0)
            break

    for pattern in time_patterns:
        match = re.search(pattern, message, re.IGNORECASE)
        if match:
            time = match.group(0)
            break

    return date, time


def extract_registration_deadline(message):
    pattern = r'((register|registration|audition|competition|deadline|last date)[^.,\n]*?(\d{1,2}(st|nd|rd|th)?\s?(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)\s?\d{0,4}|\d{1,2}/\d{1,2}/\d{2,4}))'
    match = re.search(pattern, message, re.IGNORECASE)
    return match.group(0) if match else "Not found"


def set_calendar_reminder(message, category, date, time):
    if category in ["Event", "Tech Talk", "Audition"]:
        print(f"  Reminder set for {category}!")
        print(f"   Event: {message}")
        print(f"   Date: {date}")
        print(f"   Time: {time}")
    else:
        print(f" No reminder for {category} messages")


def classify_message(message):
    X_new = vectorizer.transform([message])
    category = model.predict(X_new)[0]
    date, time = extract_datetime(message)
    registration = extract_registration_deadline(message)

    print("\nMessage:", message)
    print("Predicted Category:", category)
    print(" Event Date:", date)
    print(" Event Time:", time)
    print(" Registration / Deadline:", registration)
    print("-"*60)


    set_calendar_reminder(message, category, date, time)
    print("-"*60)

def menu():
    while True:
        print("\n Academic Message Classifier")
        print("1. Test with default sample messages")
        print("2. Enter your own message")
        print("3. Test with OCR from an image")
        print("4. Exit")
        choice = input("Choose an option (1-4): ")

        if choice == "1":
            test_messages = [
                "Cyber Security Workshop on 15th Sep 2025 at 11:30 AM",
                "Submit assignment 3 by next Wednesday",
                "Music Fest happening on 18th Sep 2025 at 6 PM",
                "Auditions for College Drama on 15th Sep at 2 PM"
            ]
            for msg in test_messages:
                classify_message(msg)

        elif choice == "2":
            msg = input("Enter your message: ")
            classify_message(msg)

        elif choice == "3":
            path = input("Enter image path: ")
            ocr_text = extract_text_from_image(path)
            print("\n OCR Extracted:", ocr_text)
            classify_message(ocr_text)

        elif choice == "4":
            print(" Exiting. Goodbye!")
            break
        else:
            print(" Invalid choice. Try again.")

if __name__ == "__main__":
    menu()


 Model trained!


 Academic Message Classifier
1. Test with default sample messages
2. Enter your own message
3. Test with OCR from an image
4. Exit
Choose an option (1-4): 2
Enter your message: tomorrow there is a dance competetion

Message: tomorrow there is a dance competetion
Predicted Category: Other Academic
 Event Date: Not found
 Event Time: Not found
 Registration / Deadline: Not found
------------------------------------------------------------
 No reminder for Other Academic messages
------------------------------------------------------------

 Academic Message Classifier
1. Test with default sample messages
2. Enter your own message
3. Test with OCR from an image
4. Exit
